# Run polygon UTrack workflows

This notebook runs the polygon moisture-flow scripts in dependency order for all HPA ERA5 grid cells. Edit the input paths and output-folder names in the configuration cell, then run all cells.

In [1]:
from pathlib import Path
import subprocess
import sys

## Configuration

Set the UTrack and corrected ERA5 directories. The polygon grid defaults to `data\ERA5_grid\ERA5_grid_HPA.shp`.

In [6]:
# Input data paths
home = Path.home()
utrack_directory = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "UTrack"
era5_directory = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "RECON"
polygon_grid_file = Path("data") / "ERA5_grid" / "ERA5_grid_HPA.shp"

# UTrack processing controls
flow_encoding = "encoded-log"
batch_size = 16

# Resolve the repository root if Jupyter starts from the repository or its code directory.
current_directory = Path.cwd().resolve()
if (current_directory / "data").is_dir() and (current_directory / "figures+tables").is_dir():
    repository_root = current_directory
elif (current_directory.parent / "data").is_dir() and (current_directory.parent / "figures+tables").is_dir():
    repository_root = current_directory.parent
else:
    raise FileNotFoundError("Start Jupyter from the repository root or its code directory.")

polygon_grid_file = repository_root / polygon_grid_file
code_directory = repository_root / "code"
polygon_engine_script = code_directory / "UTrack-Polygon_01_MonthlyP+Eshed.py"
precipitationshed_script = code_directory / "UTrack-Polygon_02_CalculateMonthlyP+Esheds.py"
map_script = code_directory / "UTrack-Polygon_03_MapMonthlyP+Esheds.py"

# NetCDF output directories
precipitationshed_output_directory = repository_root / "data" / "UTrack_polygon_monthly_precipitationshed_HPA"
evaporationshed_output_directory = repository_root / "data" / "UTrack_polygon_monthly_evaporationshed_HPA"

# Map output directories
precipitationshed_map_output_directory = repository_root / "figures+tables" / "UTrack_polygon_monthly_precipitationshed_maps_HPA"
evaporationshed_map_output_directory = repository_root / "figures+tables" / "UTrack_polygon_monthly_evaporationshed_maps_HPA"
evaporationshed_fraction_map_output_directory = repository_root / "figures+tables" / "UTrack_polygon_monthly_evaporationshed_precipitation_fraction_maps_HPA"

for path in (utrack_directory, era5_directory, polygon_grid_file, polygon_engine_script, precipitationshed_script, map_script):
    if not path.exists():
        raise FileNotFoundError(f"Required input or script not found: {path}")
for directory in (precipitationshed_output_directory, evaporationshed_output_directory, precipitationshed_map_output_directory, evaporationshed_map_output_directory, evaporationshed_fraction_map_output_directory):
    directory.mkdir(parents=True, exist_ok=True)

## Calculate polygon sheds

`01` writes both monthly shed directions. `02` then writes the dedicated monthly precipitationshed outputs used by the mapping script.

In [ ]:
engine_command = [
    sys.executable, str(polygon_engine_script),
    "--polygon-grid-file", str(polygon_grid_file),
    "--utrack-directory", str(utrack_directory),
    "--era5-directory", str(era5_directory),
    "--precipitationshed-output-directory", str(precipitationshed_output_directory),
    "--evaporationshed-output-directory", str(evaporationshed_output_directory),
    "--flow-encoding", flow_encoding,
    "--batch-size", str(batch_size),
]
subprocess.run(engine_command, check=True)

precipitationshed_command = [
    sys.executable, str(precipitationshed_script),
    "--polygon-grid-file", str(polygon_grid_file),
    "--utrack-directory", str(utrack_directory),
    "--era5-directory", str(era5_directory),
    "--output-directory", str(precipitationshed_output_directory),
    "--flow-encoding", flow_encoding,
    "--batch-size", str(batch_size),
]
subprocess.run(precipitationshed_command, check=True)

## Map polygon sheds

The mapper writes precipitationshed contribution maps plus evaporationshed contributions expressed as a percentage of total monthly precipitation in each receiving cell.

In [7]:
map_command = [
    sys.executable, str(map_script),
    "--input-directory", str(precipitationshed_output_directory),
    "--output-directory", str(precipitationshed_map_output_directory),
    "--evaporationshed-input-directory", str(evaporationshed_output_directory),
    "--era5-directory", str(era5_directory),
    "--evaporationshed-map-output-directory", str(evaporationshed_map_output_directory),
    "--evaporationshed-fraction-output-directory", str(evaporationshed_fraction_map_output_directory),
]
subprocess.run(map_command, check=True)

CompletedProcess(args=['C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\.venv\\Scripts\\python.exe', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\code\\UTrack-Polygon_03_MapMonthlyP+Esheds.py', '--input-directory', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\data\\UTrack_polygon_monthly_precipitationshed_HPA', '--output-directory', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\figures+tables\\UTrack_polygon_monthly_precipitationshed_maps_HPA', '--evaporationshed-input-directory', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\data\\UTrack_polygon_monthly_evaporationshed_HPA', '--era5-directory', 'C:\\Users\\s947z036\\OneDrive - University of Kansas\\Research\\MoistureRecycling\\RECON', '--evaporationshed-map-output-directory', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyclingRECON\\figures+tables\\UTrack_polygon_monthly_evaporationshed_maps_HPA', '--evaporationshed-fraction-output-directory', 'C:\\Users\\s947z036\\WorkGits\\MoistureRecyc